In [1]:
# packages we need:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
# install them with pip if you do not have them

Matplotlib is building the font cache; this may take a moment.


In [2]:
results_data = pd.read_csv('data/all_data.csv')

In [3]:
results_data

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Season,Division,filename
0,1993-08-14,Arsenal,Coventry,0.0,3.0,A,9394,E0,data/E0/E0_9394.csv
1,1993-08-14,Preston,Crewe,0.0,2.0,A,9394,E3,data/E3/E3_9394.csv
2,1993-08-14,Mansfield,Shrewsbury,1.0,0.0,H,9394,E3,data/E3/E3_9394.csv
3,1993-08-14,Hereford,Scarborough,0.0,1.0,A,9394,E3,data/E3/E3_9394.csv
4,1993-08-14,Gillingham,Chesterfield,0.0,2.0,A,9394,E3,data/E3/E3_9394.csv
...,...,...,...,...,...,...,...,...,...
65197,2025-09-20,Walsall,Tranmere,4.0,2.0,H,2526,E3,data/E3/E3_2526.csv
65198,2025-09-21,Bristol City,Oxford,1.0,3.0,A,2526,E1,data/E1/E1_2526.csv
65199,2025-09-21,Arsenal,Man City,1.0,1.0,D,2526,E0,data/E0/E0_2526.csv
65200,2025-09-21,Sunderland,Aston Villa,1.0,1.0,D,2526,E0,data/E0/E0_2526.csv


In [4]:
def create_league_table(data):
    
    unique_team_names = data['HomeTeam'].unique()

    league_table = pd.DataFrame()

    league_table['team'] = unique_team_names
    league_table['ranking'] = 0
    league_table['points'] = 0
    league_table['w'] = 0
    league_table['d'] = 0
    league_table['l'] = 0
    league_table['goals_for'] = 0
    league_table['goals_against'] = 0
    league_table['goal_difference'] = 0
    league_table['matched_played'] = 0
    league_table['last_date'] = 0
    league_table['ppg'] = 0

    # if there is colname 'Date', then we can use it to update 'last_date' column:
    if 'Date' in data.columns:
        league_table['last_date'] = min(data['Date'])
    else:
        league_table['last_date'] = 0
    
    return league_table

In [5]:

lt = create_league_table(results_data)


In [6]:
def update_league_table(league_table, results_table):

    league_table_all = league_table.copy()
    league_table_new = league_table.copy()

    for index, row in results_table.iterrows():
        # get team names
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']
        # get match results
        home_team_goals = row['FTHG']
        away_team_goals = row['FTAG']
        full_time_result = row['FTR']
        # get date of match
        last_date = row['Date']

        # update 'matched_played' column:
        league_table_new.loc[league_table_new['team'] == home_team, 'matched_played'] += 1
        league_table_new.loc[league_table_new['team'] == away_team, 'matched_played'] += 1

        # update 'goals_for' and 'goals_against' columns:     
        league_table_new.loc[league_table_new['team'] == home_team, 'goals_for'] += home_team_goals
        league_table_new.loc[league_table_new['team'] == home_team, 'goals_against'] += away_team_goals
        league_table_new.loc[league_table_new['team'] == away_team, 'goals_for'] += away_team_goals
        league_table_new.loc[league_table_new['team'] == away_team, 'goals_against'] += home_team_goals

        # update 'last_date' column:
        league_table_new.loc[league_table_new['team'] == home_team, 'last_date'] = last_date

        # update 'ponts' and 'w', 'd', 'l' columns based on the result of the match:
        if full_time_result == 'H':
            league_table_new.loc[league_table_new['team'] == home_team, 'points'] += 3
            league_table_new.loc[league_table_new['team'] == away_team, 'points'] += 0

            league_table_new.loc[league_table_new['team'] == home_team, 'w'] += 1
            league_table_new.loc[league_table_new['team'] == away_team, 'l'] += 1

        elif full_time_result == 'A':
            league_table_new.loc[league_table_new['team'] == home_team, 'points'] += 0
            league_table_new.loc[league_table_new['team'] == away_team, 'points'] += 3

            league_table_new.loc[league_table_new['team'] == home_team, 'l'] += 1
            league_table_new.loc[league_table_new['team'] == away_team, 'w'] += 1
            
        elif full_time_result == 'D':
            league_table_new.loc[league_table_new['team'] == home_team, 'points'] += 1
            league_table_new.loc[league_table_new['team'] == away_team, 'points'] += 1

            league_table_new.loc[league_table_new['team'] == home_team, 'd'] += 1
            league_table_new.loc[league_table_new['team'] == away_team, 'd'] += 1
        else:
            print('Error: FTR is not H, A or D')

        # update 'goal_difference' column:
        league_table_new['goal_difference'] = league_table_new['goals_for'] - league_table_new['goals_against']
        
        # update 'ppg' column:
        league_table_new['ppg'] = league_table_new['points'] / league_table_new['matched_played']

        # crate ranking based on points, goal difference, goals for and goals against
        league_table_new = league_table_new.sort_values(by=['points', 'goal_difference', 'goals_for', 'goals_against'], ascending=False)
        league_table_new = league_table_new.reset_index(drop=True)
        league_table_new['ranking'] = league_table_new.index + 1
        
        # update our data
        updated_row = league_table_new[(league_table_new['team'] == home_team) | (league_table_new['team'] == away_team)]
        league_table_all = pd.concat([league_table_all, updated_row], ignore_index=True)

    # reorder rows based on column points and goals_for:
    league_table_new = league_table_new.sort_values(by=['ppg','points', 'goal_difference', 'goals_for', 'goals_against'], ascending=False)
    league_table_new = league_table_new.reset_index(drop=True)
    
    return league_table_new, league_table_all

In [7]:
results = results_data
table_data = create_league_table(results)
new_table_data, league_table_all = update_league_table(table_data, results)


In [8]:
new_table_data.to_csv('data/all_table.csv', index=False)


In [9]:
new_table_data

,team,ranking,points,w,d,l,goals_for,goals_against,goal_difference,matched_played,last_date,ppg
0,Man United,1,2466,733,267,229,2283,1198,1085,1229,2025-09-20,2.006509
1,Arsenal,2,2342,681,299,249,2235,1218,1017,1229,2025-09-21,1.905614
2,Liverpool,3,2298,666,300,263,2217,1226,991,1229,2025-09-20,1.869813
3,Chelsea,4,2266,655,301,273,2111,1229,882,1229,2025-08-30,1.843775
4,Man City,5,2209,645,274,350,2243,1374,869,1269,2025-09-14,1.740741
...,...,...,...,...,...,...,...,...,...,...,...,...
111,Morecambe,91,969,249,222,348,994,1217,-223,819,2025-05-03,1.183150
112,Macclesfield,94,910,229,223,321,869,1068,-199,773,2020-02-29,1.177232
113,Chester,101,643,162,157,229,624,757,-133,548,2009-05-02,1.173358
114,Scarborough,107,310,80,70,118,326,403,-77,268,1999-05-08,1.156716
